####Package Installation

In [1]:
!pip install faiss-cpu langchain langchain-community langchain-google-genai pandas

####Imports

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"]=userdata.get('GOOGLE_API_KEY')

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

####Initiate the Vector Store

In [4]:
pip install -qU langchain-huggingface

In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
index = faiss.IndexFlatL2(len(embeddings.embed_query(" ")))
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

####Create a loader for JSON files

In [3]:
pip install jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.1/757.1 kB 20.9 MB/s eta 0:00:00


In [7]:
from langchain_community.document_loaders import JSONLoader
file_path = 'app_insights.json'

# Define the jq schema to extract data from the JSON.
# This example assumes you want to load the entire JSON content as a single document.
# You may need to adjust the jq_schema based on the structure of your JSON file
# to extract specific fields or arrays.
# For example, to load a list of objects under a key 'items', you might use jq_schema=".items[]"
# To load a specific field 'answer', you might use jq_schema=".answer"
loader = JSONLoader(
    file_path=file_path,
    jq_schema='.', # This loads the entire JSON object
    text_content=False) # Set to True if the content you want to load is directly under the jq_schema path and is text

docs = loader.load()

####Process CSV Data

In [ ]:
loader = CSVLoader(file_path='cleaned_googleplaystore.csv')
docs = loader.load_and_split()
##line docs = loader.load_and_split() reads your CSV file and creates a list called docs, where each element in the list represents a row from your CSV file, formatted as a document object that can be used in subsequent steps for tasks like creating embeddings and building a vector store.

####Add the splitted csv data to the vector store

In [8]:
vector_store.add_documents(documents=docs)

['fee6be04-4219-4f35-b067-cf649314f7ac']

####Create teh Retrieval Chain

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retriever = vector_store.as_retriever()

# Set up system prompt
system_prompt = (
    """
    You are a helpful chatbot.
    """
    "{context}"
    ###update the system prompt on however you like it to work - either for query insights or querying dataset
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),

])

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

#### Chatbot Interface

In [10]:
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        break
    response = rag_chain.invoke({"input": user_input})
    print(f"Bot: {response['answer']}")

You: hi
Bot: Hello! How can I help you further with the app data analysis or anything else?
You: yes tell me something about the app insights
Bot: Based on the dataset you provided, here are the key app insights:

1.  **Free is King for Adoption:** Apps offered for free (`Type: Free`, `Price($): 0.0`) consistently achieve significantly higher install numbers (e.g., 100,000+ for DataCamp and Microsoft Power BI) compared to paid apps, even for specialized tools.
2.  **Education & Business Drive Engagement:** Apps focused on learning (R, Python, SQL, Business Intelligence) or business utility (Power BI) tend to garner high ratings (4.4-4.9) and substantial user reviews, indicating strong user satisfaction and perceived value.
3.  **High Price = Low Adoption for Niche Apps:** The "Dr.Dice - Sic bo analyzer" app, priced at $46.99, shows extremely low installs (10) and minimal reviews (2). This highlights a significant barrier to entry for highly-priced niche apps without an established bran